In [1]:
# 修正后的交叉验证评估（只用于输出正确的标准差）
# 这里不保存任何模型，不改变您已有的 .joblib 文件
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import catboost as cb

# 1. Load datasets
# ----------------------------------------------------------------------
file_path = "Global378+Tibet15+Fossil73_更正.xlsx"

# Modern calibration dataset (393 sites: 378 global + 15 Tibetan Plateau)
df_modern = pd.read_excel(file_path, sheet_name="plants")
# Fossil dataset (73 assemblages)
df_fossil = pd.read_excel(file_path, sheet_name="fossils")

print(f"Modern dataset shape: {df_modern.shape}")
print(f"Fossil dataset shape: {df_fossil.shape}")

feature_cols = [
    'Lobed', 'No Teeth', 'Teeth Regul', 'Teeth Close', 'Teeth Round', 'Teeth Acute', 'Teeth Compo',
    'Nanophyl', 'Leptophyl', 'Leptophyl 2', 'Microphyl 1', 'Microphyl 2', 'Microphyl 3',
    'Mesophyl 1', 'Mesophyl 2', 'Mesophyl 3', 'Apex Emargi', 'Apex Round', 'Apex Acute', 'Apex Attenu',
    'Base Cordat', 'Base Round', 'Base Acute', 'L:W <1:1', 'L:W 1-2:1', 'L:W 2-3:1', 'L:W 3-4:1',
    'L:W >4:1', 'Obovate', 'Elliptic', 'Ovate'
]

target_cols = ['MAT', 'WMMT', 'CMMT', 'GROWSEAS', 'RH', 'SH', 'ENTHAL']
# 3. Data cleaning (drop any rows with missing values)
# ----------------------------------------------------------------------
modern_features = df_modern[feature_cols].copy()
modern_targets = df_modern[target_cols].copy()
mask = modern_features.isna().any(axis=1) | modern_targets.isna().any(axis=1)
if mask.any():
    print(f"Dropping {mask.sum()} rows with missing values in modern dataset.")
    modern_features = modern_features[~mask]
    modern_targets = modern_targets[~mask]

print(f"Clean modern dataset: {modern_features.shape[0]} samples")

# For fossil data: keep only available features (should be same columns)
fossil_features = df_fossil[feature_cols].copy()
if fossil_features.isna().any().any():
    print("Warning: Fossil dataset contains missing values. Filling with column median.")
    fossil_features = fossil_features.fillna(fossil_features.median())

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
ada = AdaBoostRegressor(n_estimators=200, random_state=42)

# XGBoost with GPU
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    random_state=42,
    verbosity=0,
    tree_method= 'hist',           # 新版本推荐使用 hist 配合
    device='cuda',                # 或者 'cuda:0'
    predictor= 'auto'
)

# CatBoost with GPU
cat = cb.CatBoostRegressor(
    n_estimators=200,
    random_seed=42,
    verbose=False,
    task_type='CPU',
    devices='0'
)

# Stacking ensemble (base learners + linear meta-learner)
stack = StackingRegressor(
    estimators=[('rf', rf), ('ada', ada), ('xgb', xgb_model), ('cat', cat)],
    final_estimator=LinearRegression(),
    cv=5
)

models = {
    'Random Forest': rf,
    'AdaBoost': ada,
    'XGBoost': xgb_model,
    'CatBoost': cat,
    'Stacking': stack
}
# 4. Define evaluation metrics
# ----------------------------------------------------------------------
def mean_absolute_percentage_error(y_true, y_pred, eps=1e-6):
    """Mean absolute percentage error, safe for near-zero values."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = np.abs(y_true) + eps
    return np.mean(np.abs((y_true - y_pred) / denominator)) * 100

def evaluate_model(y_true, y_pred):
    """Compute R², RMSE, MAE, MAPE."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)
results = {t: {name: {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE': []} for name in models} for t in target_cols}

for target in target_cols:
    y = modern_targets[target].values
    X = modern_features.values
    for name, model in models.items():
        fold_scores = []  # 存储每一折的指标
        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            model_clone = clone(model)
            model_clone.fit(X_train, y_train)
            y_pred = model_clone.predict(X_test)
            r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae = mean_absolute_error(y_test, y_pred)
            mape = mean_absolute_percentage_error(y_test, y_pred)  # 函数需要定义
            fold_scores.append({'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape})
        for metric in ['R2', 'RMSE', 'MAE', 'MAPE']:
            vals = [s[metric] for s in fold_scores]
            results[target][name][metric] = (np.mean(vals), np.std(vals))
    print(f"Finished {target}")

# 打印表格（现在标准差非零）

Modern dataset shape: (393, 47)
Fossil dataset shape: (71, 47)
Clean modern dataset: 393 samples
Finished MAT
Finished WMMT
Finished CMMT
Finished GROWSEAS
Finished RH
Finished SH
Finished ENTHAL
